
# Experiment C — Wavelet Frequency Mask Ablation

This notebook measures the **conditional importance of individual wavelet-frequency bands** by masking event channels **before acceleration reconstruction**.

## Experimental design

You provide **one input combination** (one or more dataset roots, typically Action0 + Action1) whose metadata defines exactly five wavelet bands. The notebook then:

1. reads the five frequencies from `spike_encoder.frequencies_hz`;
2. runs a **full 5-band baseline** with no masking;
3. masks the non-backbone fifth band and runs the **`(1, 2, 4, 8)` backbone baseline**;
4. automatically creates leave-one-frequency-out backbone ablations:
   - keep `(2, 4, 8)` → remove `1 Hz`;
   - keep `(1, 4, 8)` → remove `2 Hz`;
   - keep `(1, 2, 8)` → remove `4 Hz`;
   - keep `(1, 2, 4)` → remove `8 Hz`;
5. reconstructs acceleration from the masked event channels;
6. evaluates every condition with the same fixed user split, the same five training seeds, and `cnn_s / cnn_m / cnn_l`;
7. reports absolute performance and **seed-paired ablation loss relative to the `(1,2,4,8)` backbone**.

The mask is applied to the 15 event channels using the repository convention **axis-major, frequency-minor** (`3 axes × 5 bands`). Raw IMU channels are never changed.

### Interpretation

For metric \(M\), the leave-one-out importance of frequency \(f\) is

\[
I_f = M(1,2,4,8) - M((1,2,4,8)\\setminus f).
\]

A positive value means that removing the frequency hurts downstream performance under the fixed backbone context.


In [ ]:

from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import sys
from dataclasses import replace
from pathlib import Path
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "run_experiment_c.py").is_file() and (candidate / "snn").is_dir():
            return candidate
    raise RuntimeError("Could not locate the writingRing repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.run_experiment_c import run_experiment_c
from scripts import reconstruct_padded_spike_accel as recon
from snn.accel_reconstruction_eval import (
    experiment_c_config,
    load_acceleration_data,
    normalize_user_name,
    prepare_user_disjoint_splits,
)

print("Repository root:", REPO_ROOT)



## 1. Settings

Edit `INPUT_COMBINATION_ROOTS` only. It may contain one dataset root or multiple compatible roots (for example Action0 + Action1). Every selected root must have the same five-band Custom Wavelet encoder metadata.

The input must still contain the original variable-length `segmentation/` data because masking is applied to unpadded event sequences before reconstruction.

`INCLUDED_LABELS` is applied after user exclusion and controls the final train/validation/test cohort for every ablation condition.


In [ ]:

# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------

INPUT_COMBINATION_ROOTS: tuple[str, ...] = (
    # Example:
    "outputs/action0_default_wavelets/low-pass/aligned-board-events",
    "outputs/action1_default_wavelets/low-pass/aligned-board-events",
)

# Scientific backbone under test.
BACKBONE_FREQUENCIES_HZ: tuple[float, ...] = (1.0, 2.0, 4.0, 8.0)

# Same design as the frequency benchmark notebook.
PROBES: tuple[str, ...] = ("cnn_s", "cnn_m", "cnn_l")
TRAINING_SEEDS: tuple[int, ...] = (13, 37, 71, 101, 137)
SPLIT_SEED: int = 12345

EXCLUDED_USERS: tuple[str, ...] = ("user_17",)
# Ordered labels to retain after user exclusion; None keeps every label.
INCLUDED_LABELS: tuple[str, ...] | None = ("A", "B", "C", "D", "E", "X", "G", "H", "I", "J", "K", "L")
TRAIN_FRACTION: float = 0.70
VAL_FRACTION: float = 0.15
REQUIRE_ALL_LABELS_IN_ALL_SPLITS: bool = True

PRIMARY_METRICS: dict[str, str] = {
    "CNN balanced accuracy": "CNN_test_balanced_accuracy",
    "CNN macro-F1": "CNN_test_macro_f1",
    "kNN balanced accuracy": "kNN_test_balanced_accuracy",
    "Linear probe balanced accuracy": "linear_probe_test_balanced_accuracy",
    "Retrieval macro mAP": "retrieval_macro_mAP",
}

SUPPLEMENTARY_METRICS: dict[str, str] = {
    "SameLabel@1 macro": "SameLabel_macro_at_1",
    "D intra": "D_intra_macro",
    "D inter": "D_inter_centroids",
    "D inter / D intra": "D_inter_over_D_intra",
    "Silhouette macro": "silhouette_macro",
}

OUTPUT_ROOT = REPO_ROOT / "notebooks/artifacts/experiment_C_frequency_ablation"
DERIVED_DATASET_ROOT = OUTPUT_ROOT / "masked_datasets"
RUN_ROOT = OUTPUT_ROOT / "runs"
FIGURE_ROOT = OUTPUT_ROOT / "figures"

# Rebuild masked reconstructions even if derived datasets already exist.
REBUILD_MASKED_DATASETS: bool = False

# Reuse completed Experiment C runs when summary.csv + embeddings_test.npz exist.
RESUME: bool = True

# Set True only after the input and preflight tables look correct.
RUN_BENCHMARK: bool = False

# None lets run_experiment_c choose CUDA when configured/available.
DEVICE: str | None = None

for path in (OUTPUT_ROOT, DERIVED_DATASET_ROOT, RUN_ROOT, FIGURE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

if not INPUT_COMBINATION_ROOTS:
    print("Set INPUT_COMBINATION_ROOTS before running the dataset-construction cells.")



## 2. Resolve the input combination and read the five wavelet frequencies

The five frequencies are read from each padded dataset's `padding_dataset_summary.json → spike_encoder → frequencies_hz`. All selected roots must agree exactly (within floating-point tolerance).


In [ ]:

FREQ_ATOL = 1e-9


def fmt_freq(value: float) -> str:
    value = float(value)
    if value.is_integer():
        return str(int(value))
    return f"{value:g}"


def freq_tuple_label(values: Sequence[float]) -> str:
    return "_".join(fmt_freq(v) for v in values)


def same_frequency(a: float, b: float) -> bool:
    return math.isclose(float(a), float(b), rel_tol=0.0, abs_tol=FREQ_ATOL)


def resolve_combination_root(value: str | Path) -> tuple[Path, Path]:
    root = Path(value).expanduser()
    if not root.is_absolute():
        root = REPO_ROOT / root
    root = root.resolve()

    # Combination root containing segmentation/ and segmentation_padded/.
    if (root / "segmentation").is_dir() and (root / "segmentation_padded").is_dir():
        combination_root = root
        padded_root = root / "segmentation_padded"
    # Direct padded root.
    elif root.name == "segmentation_padded" and (root / "padding_dataset_summary.json").is_file():
        combination_root = root.parent
        padded_root = root
    else:
        raise FileNotFoundError(
            f"Expected a combination root with segmentation/ + segmentation_padded/, "
            f"or a direct segmentation_padded root: {root}"
        )

    if not (combination_root / "segmentation").is_dir():
        raise FileNotFoundError(
            f"Mask reconstruction requires the original variable-length segmentation/: {combination_root}"
        )
    if not (padded_root / "padding_dataset_summary.json").is_file():
        raise FileNotFoundError(f"Missing padding_dataset_summary.json: {padded_root}")
    return combination_root, padded_root


def read_encoder_frequencies(padded_root: Path) -> tuple[float, ...]:
    summary_path = padded_root / "padding_dataset_summary.json"
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    spec = summary.get("spike_encoder")
    if not isinstance(spec, dict):
        raise ValueError(f"{summary_path}: missing spike_encoder metadata")
    raw = spec.get("frequencies_hz")
    if not isinstance(raw, list) or len(raw) != 5:
        raise ValueError(f"{summary_path}: frequencies_hz must contain exactly five values")
    frequencies = tuple(float(v) for v in raw)
    if any(not math.isfinite(v) or v <= 0 for v in frequencies):
        raise ValueError(f"{summary_path}: invalid frequencies_hz={frequencies}")
    return frequencies


RESOLVED_INPUTS: list[dict[str, object]] = []
INPUT_FREQUENCIES_HZ: tuple[float, ...] | None = None

for root_index, root_value in enumerate(INPUT_COMBINATION_ROOTS):
    combination_root, padded_root = resolve_combination_root(root_value)
    frequencies = read_encoder_frequencies(padded_root)
    if INPUT_FREQUENCIES_HZ is None:
        INPUT_FREQUENCIES_HZ = frequencies
    elif len(frequencies) != len(INPUT_FREQUENCIES_HZ) or any(
        not same_frequency(a, b) for a, b in zip(frequencies, INPUT_FREQUENCIES_HZ, strict=True)
    ):
        raise ValueError(
            "All roots in one input combination must use the same five wavelet frequencies. "
            f"Expected {INPUT_FREQUENCIES_HZ}, got {frequencies} at {padded_root}"
        )
    RESOLVED_INPUTS.append(
        {
            "root_index": root_index,
            "combination_root": combination_root,
            "padded_root": padded_root,
            "frequencies_hz": frequencies,
        }
    )

if INPUT_FREQUENCIES_HZ is not None:
    missing_backbone = [
        f for f in BACKBONE_FREQUENCIES_HZ
        if not any(same_frequency(f, candidate) for candidate in INPUT_FREQUENCIES_HZ)
    ]
    if missing_backbone:
        raise ValueError(
            f"Input encoder {INPUT_FREQUENCIES_HZ} does not contain the full backbone "
            f"{BACKBONE_FREQUENCIES_HZ}; missing {missing_backbone}"
        )

    extra = [
        f for f in INPUT_FREQUENCIES_HZ
        if not any(same_frequency(f, b) for b in BACKBONE_FREQUENCIES_HZ)
    ]
    if len(extra) != 1:
        raise ValueError(
            "Expected exactly one fifth frequency outside the fixed backbone "
            f"{BACKBONE_FREQUENCIES_HZ}; input={INPUT_FREQUENCIES_HZ}, extra={extra}"
        )
    EXTRA_FREQUENCY_HZ = float(extra[0])
    print("Input five bands:", INPUT_FREQUENCIES_HZ)
    print("Fixed backbone:", BACKBONE_FREQUENCIES_HZ)
    print("Extra fifth band:", EXTRA_FREQUENCY_HZ)
else:
    EXTRA_FREQUENCY_HZ = None



## 3. Automatically build the ablation conditions

The leave-one-out conditions always mask the extra fifth band first, then remove one member of the `(1,2,4,8)` backbone. This makes every importance score conditional on the same four-band backbone definition.


In [ ]:

def sorted_by_input_order(values: Iterable[float]) -> tuple[float, ...]:
    if INPUT_FREQUENCIES_HZ is None:
        return tuple(float(v) for v in values)
    selected = tuple(float(v) for v in values)
    return tuple(
        f for f in INPUT_FREQUENCIES_HZ
        if any(same_frequency(f, v) for v in selected)
    )


def condition_key(prefix: str, kept: Sequence[float]) -> str:
    return f"{prefix}__keep_{freq_tuple_label(kept)}Hz"


CONDITIONS: list[dict[str, object]] = []
if INPUT_FREQUENCIES_HZ is not None:
    full_kept = tuple(INPUT_FREQUENCIES_HZ)
    backbone_kept = sorted_by_input_order(BACKBONE_FREQUENCIES_HZ)

    CONDITIONS.append(
        {
            "condition": condition_key("full", full_kept),
            "role": "full_5band_baseline",
            "kept_frequencies_hz": full_kept,
            "masked_frequencies_hz": tuple(),
            "removed_backbone_frequency_hz": np.nan,
        }
    )
    CONDITIONS.append(
        {
            "condition": condition_key("backbone", backbone_kept),
            "role": "backbone_baseline",
            "kept_frequencies_hz": backbone_kept,
            "masked_frequencies_hz": tuple(
                f for f in INPUT_FREQUENCIES_HZ
                if not any(same_frequency(f, k) for k in backbone_kept)
            ),
            "removed_backbone_frequency_hz": np.nan,
        }
    )

    for removed in BACKBONE_FREQUENCIES_HZ:
        kept = sorted_by_input_order(
            f for f in BACKBONE_FREQUENCIES_HZ if not same_frequency(f, removed)
        )
        masked = tuple(
            f for f in INPUT_FREQUENCIES_HZ
            if not any(same_frequency(f, k) for k in kept)
        )
        CONDITIONS.append(
            {
                "condition": condition_key(f"remove_{fmt_freq(removed)}Hz", kept),
                "role": "leave_one_backbone_frequency_out",
                "kept_frequencies_hz": kept,
                "masked_frequencies_hz": masked,
                "removed_backbone_frequency_hz": float(removed),
            }
        )

CONDITION_TABLE = pd.DataFrame(CONDITIONS)
if not CONDITION_TABLE.empty:
    CONDITION_TABLE["kept_frequencies_hz"] = CONDITION_TABLE["kept_frequencies_hz"].map(list)
    CONDITION_TABLE["masked_frequencies_hz"] = CONDITION_TABLE["masked_frequencies_hz"].map(list)
    display(CONDITION_TABLE)



## 4. Construct derived padded datasets with masked reconstructions

The original dataset is never modified. For each condition the notebook creates a lightweight derived `segmentation_padded` tree:

- unchanged padded SpikeIMU / labels / masks / manifests are hard-linked when possible (copied otherwise);
- existing reconstruction files are skipped;
- reconstruction is regenerated from the **original unpadded event sequence** after zeroing masked frequency bands;
- reconstruction metadata records the kept and masked frequencies.

Because the event layout is axis-major/frequency-minor, a `(T,15)` event matrix is reshaped to `(T,3,5)` and masking is performed on the last dimension.


In [ ]:

def link_or_copy_file(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        return
    try:
        os.link(source, target)
    except OSError:
        shutil.copy2(source, target)


def populate_derived_padded_tree(source_padded_root: Path, target_padded_root: Path) -> None:
    if REBUILD_MASKED_DATASETS and target_padded_root.exists():
        shutil.rmtree(target_padded_root)
    target_padded_root.mkdir(parents=True, exist_ok=True)

    for source in source_padded_root.rglob("*"):
        relative = source.relative_to(source_padded_root)
        target = target_padded_root / relative
        if source.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        if source.name.endswith(recon.OUTPUT_SUFFIX) or source.name.endswith(recon.METADATA_SUFFIX):
            continue
        link_or_copy_file(source, target)


def frequency_indices(all_frequencies: Sequence[float], kept_frequencies: Sequence[float]) -> tuple[int, ...]:
    indices: list[int] = []
    for kept in kept_frequencies:
        matches = [i for i, f in enumerate(all_frequencies) if same_frequency(f, kept)]
        if len(matches) != 1:
            raise ValueError(
                f"Frequency {kept:g} Hz must match exactly one source band; source={all_frequencies}"
            )
        indices.append(matches[0])
    return tuple(indices)


def exported_segment_rows(manifest_path: Path) -> list[tuple[int, int, int]]:
    rows = recon.load_padding_manifest(manifest_path)
    exported: list[tuple[int, int, int]] = []
    for row in rows:
        if not recon.parse_bool(row["exported"]):
            continue
        exported.append(
            (
                recon.parse_manifest_int(row["segment_index"], field="segment_index", path=manifest_path),
                recon.parse_manifest_int(row["output_segment_index"], field="output_segment_index", path=manifest_path),
                recon.parse_manifest_int(row["original_length"], field="original_length", path=manifest_path),
            )
        )
    exported.sort(key=lambda item: item[1])
    return exported


def build_masked_package_reconstruction(
    *,
    combination_root: Path,
    source_padded_root: Path,
    target_padded_root: Path,
    user: str,
    action: str,
    source_dir: Path,
    stem: str,
    kept_frequencies_hz: Sequence[float],
    expected_frequencies_hz: Sequence[float],
) -> dict[str, object]:
    spike_path = source_dir / f"{stem}_spikeIMU.npy"
    offsets_path = source_dir / f"{stem}_segment_offsets.npy"
    lengths_path = source_dir / f"{stem}_segment_lengths.npy"
    source_summary_path = source_dir / f"{stem}_segmentation_summary.json"

    source_padded_dir = source_padded_root / user / f"action_{action}"
    target_padded_dir = target_padded_root / user / f"action_{action}"
    padded_spike_path = source_padded_dir / f"{stem}_paddedSpikeIMU.npy"
    valid_lengths_path = source_padded_dir / f"{stem}_valid_lengths.npy"
    valid_mask_path = source_padded_dir / f"{stem}_valid_mask.npy"
    padding_manifest_path = source_padded_dir / f"{stem}_padding_manifest.csv"

    required = [
        spike_path,
        offsets_path,
        lengths_path,
        source_summary_path,
        padded_spike_path,
        valid_lengths_path,
        valid_mask_path,
        padding_manifest_path,
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Missing required package files: " + ", ".join(missing))

    output_path = target_padded_dir / f"{stem}{recon.OUTPUT_SUFFIX}"
    metadata_path = target_padded_dir / f"{stem}{recon.METADATA_SUFFIX}"
    if output_path.is_file() and metadata_path.is_file() and not REBUILD_MASKED_DATASETS:
        return {
            "user": user,
            "action": action,
            "stem": stem,
            "status": "reused",
            "output_path": output_path,
        }

    spike_imu = np.load(spike_path, allow_pickle=False, mmap_mode="r")
    offsets = np.load(offsets_path, allow_pickle=False).astype(np.int64)
    lengths = np.load(lengths_path, allow_pickle=False).astype(np.int64)
    padded_spike = np.load(padded_spike_path, allow_pickle=False, mmap_mode="r")
    valid_lengths = np.load(valid_lengths_path, allow_pickle=False).astype(np.int64)
    valid_mask = np.load(valid_mask_path, allow_pickle=False, mmap_mode="r")

    source_summary = recon.load_json(source_summary_path)
    _, encoder_hash, frequencies_hz, wavelet_widths_samples = recon.load_encoder_contract(
        source_summary, path=source_summary_path
    )
    if len(frequencies_hz) != len(expected_frequencies_hz) or any(
        not same_frequency(a, b) for a, b in zip(frequencies_hz, expected_frequencies_hz, strict=True)
    ):
        raise ValueError(
            f"{source_summary_path}: package frequency metadata {frequencies_hz} "
            f"does not match input combination {tuple(expected_frequencies_hz)}"
        )

    keep_indices = frequency_indices(frequencies_hz, kept_frequencies_hz)
    keep_index_set = set(keep_indices)
    masked_indices = tuple(i for i in range(len(frequencies_hz)) if i not in keep_index_set)
    masked_frequencies_hz = tuple(frequencies_hz[i] for i in masked_indices)

    if spike_imu.ndim != 2 or spike_imu.shape[1] != 21:
        raise ValueError(f"{spike_path}: expected (N,21), got {spike_imu.shape}")
    if padded_spike.ndim != 3 or padded_spike.shape[2] != 21:
        raise ValueError(f"{padded_spike_path}: expected (S,T,21), got {padded_spike.shape}")

    if offsets.ndim != 1 or len(offsets) != len(lengths) + 1:
        raise ValueError(f"{offsets_path}: expected segment_count+1 offsets")
    if int(offsets[0]) != 0 or int(offsets[-1]) != len(spike_imu):
        raise ValueError(f"{offsets_path}: offsets do not span SpikeIMU")
    if not np.array_equal(np.diff(offsets), lengths):
        raise ValueError(f"{lengths_path}: lengths != diff(offsets)")

    retained_count, target_length, _ = padded_spike.shape
    exported = exported_segment_rows(padding_manifest_path)
    if len(exported) != retained_count:
        raise ValueError(
            f"{padding_manifest_path}: exported rows={len(exported)} != retained={retained_count}"
        )
    if [row[1] for row in exported] != list(range(retained_count)):
        raise ValueError(f"{padding_manifest_path}: output_segment_index is not contiguous")

    kernels = recon.reconstruction_kernels(
        wavelet_widths_samples=wavelet_widths_samples,
        frequencies_hz=frequencies_hz,
    )
    reconstructed = np.zeros((retained_count, target_length, 3), dtype=np.float64)

    for source_index, output_index, original_length in exported:
        if int(lengths[source_index]) != original_length:
            raise ValueError(f"{stem}: source length mismatch at segment {source_index}")
        if int(valid_lengths[output_index]) != original_length:
            raise ValueError(f"{stem}: valid_length mismatch at output {output_index}")
        expected_mask = np.arange(target_length) < original_length
        if not np.array_equal(np.asarray(valid_mask[output_index], dtype=bool), expected_mask):
            raise ValueError(f"{stem}: non-canonical valid mask at output {output_index}")

        start = int(offsets[source_index])
        stop = int(offsets[source_index + 1])
        events = np.asarray(spike_imu[start:stop, :recon.EVENT_CHANNEL_COUNT], dtype=np.float64).copy()
        events_3d = events.reshape(len(events), 3, recon.EVENTS_PER_AXIS)
        if masked_indices:
            events_3d[:, :, list(masked_indices)] = 0.0
        masked_events = events_3d.reshape(len(events), recon.EVENT_CHANNEL_COUNT)

        rec_values = recon.reconstruct_segment_events(masked_events, kernels=kernels)
        reconstructed[output_index, :original_length] = rec_values

    if not np.isfinite(reconstructed).all():
        raise FloatingPointError(f"{stem}: masked reconstruction contains non-finite values")
    if retained_count and np.any(reconstructed[~np.asarray(valid_mask, dtype=bool)] != 0.0):
        raise ValueError(f"{stem}: masked reconstruction padding is not exact zero")

    target_padded_dir.mkdir(parents=True, exist_ok=True)
    recon.atomic_save_npy(output_path, reconstructed)

    metadata = {
        "schema_version": 1,
        "artifact_type": "padded_segmentwise_custom_wavelet_acceleration_reconstruction",
        "identity": {"user": user, "action": action, "package_prefix": stem},
        "source": {
            "combination_root": str(combination_root),
            "variable_spike_imu": str(spike_path),
            "source_padded_spike_imu": str(padded_spike_path),
            "spike_encoder_spec_sha256": encoder_hash,
        },
        "reconstruction": {
            "input_event_slice": [0, 15],
            "event_channel_order": "axis-major_frequency-minor",
            "method": "custom_wavelet_event_convolution_sum_with_frequency_mask",
            "source_frequencies_hz": list(frequencies_hz),
            "wavelet_widths_samples": list(wavelet_widths_samples),
            "kept_frequency_indices": list(keep_indices),
            "kept_frequencies_hz": [float(v) for v in kept_frequencies_hz],
            "masked_frequency_indices": list(masked_indices),
            "masked_frequencies_hz": [float(v) for v in masked_frequencies_hz],
            "mask_value": 0.0,
            "mask_applied_before_reconstruction": True,
            "scale_divisor": recon.DEFAULT_SCALE_DIVISOR,
            "standard_gravity_m_s2": recon.STANDARD_GRAVITY_M_S2,
        },
        "padding": {
            "target_length": int(target_length),
            "padding_side": "right",
            "padding_value_m_s2": 0.0,
            "retained_segment_count": int(retained_count),
        },
        "output": {
            "path": str(output_path),
            "shape": [int(v) for v in reconstructed.shape],
            "dtype": str(reconstructed.dtype),
            "channel_names": [
                "reconstructed_acceleration_x_m_s2",
                "reconstructed_acceleration_y_m_s2",
                "reconstructed_acceleration_z_m_s2",
            ],
            "units": ["m/s^2", "m/s^2", "m/s^2"],
        },
    }
    recon.atomic_write_text(metadata_path, json.dumps(metadata, indent=2, sort_keys=True))

    return {
        "user": user,
        "action": action,
        "stem": stem,
        "status": "built",
        "output_path": output_path,
    }


def build_condition_datasets(condition: dict[str, object]) -> tuple[Path, ...]:
    if INPUT_FREQUENCIES_HZ is None:
        raise RuntimeError("Input frequencies have not been resolved")
    condition_name = str(condition["condition"])
    kept = tuple(float(v) for v in condition["kept_frequencies_hz"])
    derived_roots: list[Path] = []

    for item in RESOLVED_INPUTS:
        root_index = int(item["root_index"])
        combination_root = Path(item["combination_root"])
        source_padded_root = Path(item["padded_root"])
        target_padded_root = DERIVED_DATASET_ROOT / condition_name / f"root_{root_index:02d}"
        populate_derived_padded_tree(source_padded_root, target_padded_root)

        package_records = []
        source_packages = recon.discover_source_packages(combination_root / "segmentation")
        for user, action, source_dir, stem in source_packages:
            package_records.append(
                build_masked_package_reconstruction(
                    combination_root=combination_root,
                    source_padded_root=source_padded_root,
                    target_padded_root=target_padded_root,
                    user=user,
                    action=action,
                    source_dir=source_dir,
                    stem=stem,
                    kept_frequencies_hz=kept,
                    expected_frequencies_hz=INPUT_FREQUENCIES_HZ,
                )
            )
        derived_roots.append(target_padded_root)
        print(condition_name, f"root_{root_index:02d}", pd.Series([r["status"] for r in package_records]).value_counts().to_dict())

    return tuple(derived_roots)


DERIVED_ROOTS_BY_CONDITION: dict[str, tuple[Path, ...]] = {}
if CONDITIONS:
    for condition in CONDITIONS:
        name = str(condition["condition"])
        DERIVED_ROOTS_BY_CONDITION[name] = build_condition_datasets(condition)

    print("Built/reused", len(DERIVED_ROOTS_BY_CONDITION), "ablation conditions")



### 4.1 Sanity check: the unmasked full baseline should match the existing reconstruction

If the original input root already contains standard reconstruction artifacts, compare them with the newly generated **full 5-band, no-mask** reconstruction. The maximum absolute difference should be zero or numerical round-off only. This verifies that the ablation path changes reconstruction only when a mask is applied.


In [ ]:

FULL_RECON_PARITY_ROWS: list[dict[str, object]] = []

full_condition_for_parity = next((c for c in CONDITIONS if c["role"] == "full_5band_baseline"), None)

if full_condition_for_parity is not None:
    full_name = str(full_condition_for_parity["condition"])
    full_derived_roots = DERIVED_ROOTS_BY_CONDITION[full_name]
    for input_item, derived_root in zip(RESOLVED_INPUTS, full_derived_roots, strict=True):
        source_padded_root = Path(input_item["padded_root"])
        for derived_recon_path in sorted(derived_root.rglob(f"*{recon.OUTPUT_SUFFIX}")):
            relative = derived_recon_path.relative_to(derived_root)
            source_recon_path = source_padded_root / relative
            if not source_recon_path.is_file():
                continue
            derived_values = np.load(derived_recon_path, allow_pickle=False, mmap_mode="r")
            source_values = np.load(source_recon_path, allow_pickle=False, mmap_mode="r")
            if derived_values.shape != source_values.shape:
                raise AssertionError(
                    f"Full-baseline reconstruction shape mismatch: {relative}: "
                    f"derived={derived_values.shape}, source={source_values.shape}"
                )
            max_abs_diff = float(np.max(np.abs(np.asarray(derived_values) - np.asarray(source_values))))
            FULL_RECON_PARITY_ROWS.append(
                {
                    "relative_path": str(relative),
                    "shape": tuple(derived_values.shape),
                    "max_abs_diff": max_abs_diff,
                }
            )

FULL_RECON_PARITY = pd.DataFrame(FULL_RECON_PARITY_ROWS)
if not FULL_RECON_PARITY.empty:
    display(FULL_RECON_PARITY)
    print("Overall max abs difference:", FULL_RECON_PARITY["max_abs_diff"].max())
else:
    print("No original reconstruction artifacts were available for full-baseline parity checking.")



## 5. Cross-condition preflight

Every derived condition must have exactly the same logical cohort: same canonical sample IDs, labels, users, actions, and valid lengths. Only reconstructed acceleration is allowed to differ.


In [ ]:

def manifest_signature(frame: pd.DataFrame) -> pd.DataFrame:
    columns = ["sample_id", "user", "action", "label", "valid_length"]
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(f"sample_manifest missing columns: {missing}")
    return frame.loc[:, columns].sort_values("sample_id").reset_index(drop=True)


def final_cohort_manifest(frame: pd.DataFrame, *, condition_name: str) -> pd.DataFrame:
    """Return the user- and label-selected manifest used for preflight checks."""
    selected = frame.copy().reset_index(drop=True)
    if EXCLUDED_USERS:
        excluded_users = {normalize_user_name(value) for value in EXCLUDED_USERS}
        normalized_users = selected["user"].astype(str).map(normalize_user_name)
        selected = selected.loc[~normalized_users.isin(excluded_users)].reset_index(drop=True)

    if INCLUDED_LABELS is None:
        return selected

    requested_labels = tuple(str(label) for label in INCLUDED_LABELS)
    available_labels = set(selected["label"].astype(str))
    missing_labels = [label for label in requested_labels if label not in available_labels]
    if missing_labels:
        raise ValueError(
            f"{condition_name}: requested labels are absent after user exclusion: {missing_labels}"
        )
    if not requested_labels:
        raise ValueError("INCLUDED_LABELS must contain at least one label when provided")

    return selected.loc[
        selected["label"].astype(str).isin(set(requested_labels))
    ].reset_index(drop=True)


DATA_BY_CONDITION: dict[str, object] = {}
PREFLIGHT_ROWS: list[dict[str, object]] = []
REFERENCE_SIGNATURE: pd.DataFrame | None = None

for condition in CONDITIONS:
    name = str(condition["condition"])
    data = load_acceleration_data(
        DERIVED_ROOTS_BY_CONDITION[name],
        repository_root=REPO_ROOT,
        require_reconstruction=True,
    )
    signature = manifest_signature(
        final_cohort_manifest(data.sample_manifest, condition_name=name)
    )
    if REFERENCE_SIGNATURE is None:
        REFERENCE_SIGNATURE = signature
    elif not signature.equals(REFERENCE_SIGNATURE):
        raise AssertionError(f"Cross-condition cohort mismatch: {name}")

    DATA_BY_CONDITION[name] = data
    PREFLIGHT_ROWS.append(
        {
            "condition": name,
            "role": condition["role"],
            "kept_frequencies_hz": list(condition["kept_frequencies_hz"]),
            "masked_frequencies_hz": list(condition["masked_frequencies_hz"]),
            "sample_count": len(signature),
            "user_count": signature["user"].nunique(),
            "class_count": signature["label"].nunique(),
            "actions": sorted(signature["action"].astype(str).unique().tolist()),
        }
    )

PREFLIGHT_TABLE = pd.DataFrame(PREFLIGHT_ROWS)
if not PREFLIGHT_TABLE.empty:
    PREFLIGHT_TABLE.to_csv(OUTPUT_ROOT / "ablation_preflight.csv", index=False)
    display(PREFLIGHT_TABLE)



## 6. Create one fixed user split

The split is created once from the full 5-band condition using `SPLIT_SEED`. All later Experiment C runs receive these users explicitly, so the five `TRAINING_SEEDS` measure training/evaluation variation rather than user-split variation.


In [ ]:

FULL_CONDITION = next((c for c in CONDITIONS if c["role"] == "full_5band_baseline"), None)
BACKBONE_CONDITION = next((c for c in CONDITIONS if c["role"] == "backbone_baseline"), None)

FIXED_SPLIT = None
if FULL_CONDITION is not None:
    full_name = str(FULL_CONDITION["condition"])
    full_manifest = DATA_BY_CONDITION[full_name].sample_manifest
    FIXED_SPLIT = prepare_user_disjoint_splits(
        full_manifest,
        train_fraction=TRAIN_FRACTION,
        val_fraction=VAL_FRACTION,
        seed=SPLIT_SEED,
        excluded_users=EXCLUDED_USERS,
        included_labels=INCLUDED_LABELS,
        require_all_users_assigned=True,
        require_all_labels_in_all_splits=REQUIRE_ALL_LABELS_IN_ALL_SPLITS,
    )

    if INCLUDED_LABELS is not None:
        expected_labels = {str(label) for label in INCLUDED_LABELS}
        actual_labels = set(FIXED_SPLIT.sample_manifest["label"].astype(str))
        if actual_labels != expected_labels:
            raise AssertionError(
                f"Final split labels {sorted(actual_labels)} do not match "
                f"INCLUDED_LABELS {list(INCLUDED_LABELS)}"
            )

    print("Train users:", FIXED_SPLIT.train_users)
    print("Val users:  ", FIXED_SPLIT.val_users)
    print("Test users: ", FIXED_SPLIT.test_users)
    print("Classes:    ", FIXED_SPLIT.class_to_idx)
    display(FIXED_SPLIT.split_summary)
    display(FIXED_SPLIT.label_split_counts)



## 7. Persist the ablation design


In [ ]:

if FIXED_SPLIT is not None:
    design = {
        "protocol": "standalone_experiment_c_frequency_mask_ablation_v1",
        "input_roots": [str(Path(v).expanduser()) for v in INPUT_COMBINATION_ROOTS],
        "source_frequencies_hz": list(INPUT_FREQUENCIES_HZ or ()),
        "backbone_frequencies_hz": list(BACKBONE_FREQUENCIES_HZ),
        "extra_fifth_frequency_hz": EXTRA_FREQUENCY_HZ,
        "conditions": [
            {
                "condition": str(c["condition"]),
                "role": str(c["role"]),
                "kept_frequencies_hz": list(c["kept_frequencies_hz"]),
                "masked_frequencies_hz": list(c["masked_frequencies_hz"]),
                "removed_backbone_frequency_hz": (
                    None if pd.isna(c["removed_backbone_frequency_hz"])
                    else float(c["removed_backbone_frequency_hz"])
                ),
            }
            for c in CONDITIONS
        ],
        "split_seed": SPLIT_SEED,
        "training_seeds": list(TRAINING_SEEDS),
        "probes": list(PROBES),
        "excluded_users": list(EXCLUDED_USERS),
        "included_labels": None if INCLUDED_LABELS is None else list(INCLUDED_LABELS),
        "train_users": list(FIXED_SPLIT.train_users),
        "val_users": list(FIXED_SPLIT.val_users),
        "test_users": list(FIXED_SPLIT.test_users),
        "class_to_idx": FIXED_SPLIT.class_to_idx,
        "primary_metrics": PRIMARY_METRICS,
        "supplementary_metrics": SUPPLEMENTARY_METRICS,
    }
    (OUTPUT_ROOT / "ablation_design.json").write_text(
        json.dumps(design, indent=2, sort_keys=True), encoding="utf-8"
    )
    print("Saved", OUTPUT_ROOT / "ablation_design.json")



## 8. Run standalone Experiment C for every mask condition

Each condition trains from scratch on its own masked reconstruction. Reconstruction-train normalization is fitted independently for every run, while user split and class cohort remain fixed.


In [ ]:

def expected_run_dir(condition_name: str, seed: int, probe: str) -> Path:
    return RUN_ROOT / condition_name / f"seed_{seed}" / probe


def build_run_config(seed: int, probe: str, output_base: Path):
    config = experiment_c_config(
        output_dir=output_base,
        random_seed=seed,
        probe_variant=probe,
    )
    split = replace(
        config.split,
        explicit_train_users=tuple(FIXED_SPLIT.train_users),
        explicit_val_users=tuple(FIXED_SPLIT.val_users),
        explicit_test_users=tuple(FIXED_SPLIT.test_users),
        excluded_users=tuple(EXCLUDED_USERS),
        included_labels=INCLUDED_LABELS,
        require_all_users_assigned=True,
        require_all_labels_in_all_splits=REQUIRE_ALL_LABELS_IN_ALL_SPLITS,
    )
    return replace(config, split=split)


def load_existing_summary(run_dir: Path) -> dict[str, object] | None:
    summary_path = run_dir / "summary.csv"
    embeddings_path = run_dir / "embeddings_test.npz"
    manifest_path = run_dir / "sample_manifest.csv"
    if RESUME and summary_path.is_file() and embeddings_path.is_file():
        if INCLUDED_LABELS is not None:
            if not manifest_path.is_file():
                return None
            run_manifest = pd.read_csv(manifest_path)
            if "label" not in run_manifest.columns:
                return None
            expected_labels = {str(label) for label in INCLUDED_LABELS}
            actual_labels = set(run_manifest["label"].astype(str))
            if actual_labels != expected_labels:
                return None
        frame = pd.read_csv(summary_path)
        if len(frame) != 1:
            raise ValueError(f"Expected exactly one row in {summary_path}")
        return frame.iloc[0].to_dict()
    return None


RESULT_ROWS: list[dict[str, object]] = []

if RUN_BENCHMARK:
    if FIXED_SPLIT is None:
        raise RuntimeError("Run the split cell first")

    for condition in CONDITIONS:
        condition_name = str(condition["condition"])
        roots = DERIVED_ROOTS_BY_CONDITION[condition_name]
        for seed in TRAINING_SEEDS:
            for probe in PROBES:
                run_dir = expected_run_dir(condition_name, seed, probe)
                existing = load_existing_summary(run_dir)
                if existing is None:
                    output_base = RUN_ROOT / condition_name / f"seed_{seed}"
                    config = build_run_config(seed, probe, output_base)
                    result = run_experiment_c(
                        root=roots,
                        repository_root=REPO_ROOT,
                        output_dir=output_base,
                        reference_checkpoint=None,
                        config=config,
                        device=DEVICE,
                        allow_new_split=True,
                        probe_variant=probe,
                    )
                    summary = result.summary.iloc[0].to_dict()
                else:
                    summary = existing

                row = {
                    "condition": condition_name,
                    "role": condition["role"],
                    "training_seed": seed,
                    "probe": probe,
                    "removed_backbone_frequency_hz": condition["removed_backbone_frequency_hz"],
                    "kept_frequencies_hz": json.dumps(list(condition["kept_frequencies_hz"])),
                    "masked_frequencies_hz": json.dumps(list(condition["masked_frequencies_hz"])),
                    **summary,
                }
                RESULT_ROWS.append(row)
                print("done", condition_name, seed, probe)

    MASTER_RESULTS = pd.DataFrame(RESULT_ROWS)
    MASTER_RESULTS.to_csv(OUTPUT_ROOT / "master_results.csv", index=False)
else:
    if INCLUDED_LABELS is not None:
        MASTER_RESULTS = pd.DataFrame()
        print("RUN_BENCHMARK=False; existing results skipped because label selection is active; set RUN_BENCHMARK=True to run or resume the selected cohort")
    else:
        master_path = OUTPUT_ROOT / "master_results.csv"
        MASTER_RESULTS = pd.read_csv(master_path) if master_path.is_file() else pd.DataFrame()
        print("RUN_BENCHMARK=False; loaded existing results" if not MASTER_RESULTS.empty else "RUN_BENCHMARK=False; no existing master_results.csv")

if not MASTER_RESULTS.empty:
    display(MASTER_RESULTS.head())



## 9. Aggregate primary and supplementary metrics

Primary metrics answer whether the masked reconstruction remains useful for classification and simple representation readouts. Geometry metrics are retained only as supplementary diagnostics.


In [ ]:

def aggregate_metric_group(frame: pd.DataFrame, metric_map: dict[str, str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame()
    required = {"condition", "probe", "training_seed", *metric_map.values()}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise KeyError(f"master_results.csv is missing columns: {missing}")

    parts = []
    for display_name, column in metric_map.items():
        part = frame[["condition", "role", "probe", "training_seed", "removed_backbone_frequency_hz", column]].copy()
        part = part.rename(columns={column: "value"})
        part["metric"] = display_name
        part["metric_column"] = column
        parts.append(part)
    long = pd.concat(parts, ignore_index=True)
    agg = (
        long.groupby(
            ["metric", "metric_column", "condition", "role", "probe", "removed_backbone_frequency_hz"],
            as_index=False,
            dropna=False,
        )
        .agg(mean=("value", "mean"), sd=("value", "std"), n=("value", "count"))
    )
    return long, agg


PRIMARY_LONG, PRIMARY_AGG = aggregate_metric_group(MASTER_RESULTS, PRIMARY_METRICS)
SUPPLEMENTARY_LONG, SUPPLEMENTARY_AGG = aggregate_metric_group(MASTER_RESULTS, SUPPLEMENTARY_METRICS)

if not PRIMARY_AGG.empty:
    PRIMARY_AGG.to_csv(OUTPUT_ROOT / "primary_metric_aggregate.csv", index=False)
    SUPPLEMENTARY_AGG.to_csv(OUTPUT_ROOT / "supplementary_metric_aggregate.csv", index=False)
    display(PRIMARY_AGG)


In [ ]:
def condition_order() -> list[str]:
    return [str(c["condition"]) for c in CONDITIONS]


def short_condition_label(condition: dict[str, object]) -> str:
    role = str(condition["role"])
    kept = tuple(condition["kept_frequencies_hz"])
    if role == "full_5band_baseline":
        return "full\n(" + ",".join(fmt_freq(v) for v in kept) + ")"
    if role == "backbone_baseline":
        return "backbone\n(1,2,4,8)"
    removed = float(condition["removed_backbone_frequency_hz"])
    return f"remove {fmt_freq(removed)} Hz\nkeep (" + ",".join(fmt_freq(v) for v in kept) + ")"


CONDITION_LABELS = {str(c["condition"]): short_condition_label(c) for c in CONDITIONS}


def plot_absolute_primary_metrics(
    aggregate: pd.DataFrame,
    *,
    output_dir: Path = FIGURE_ROOT / "absolute_conditions",
) -> dict[str, Path]:
    """One figure per metric; keep cnn_s/cnn_m/cnn_l in separate panels."""
    saved: dict[str, Path] = {}
    if aggregate.empty:
        return saved
    output_dir.mkdir(parents=True, exist_ok=True)
    order = condition_order()
    x = np.arange(len(order), dtype=float)

    for metric in PRIMARY_METRICS:
        subset = aggregate[aggregate["metric"] == metric]
        fig, axes = plt.subplots(
            1, len(PROBES), figsize=(5.6 * len(PROBES), 5.0), sharey=True, squeeze=False
        )
        for col, probe in enumerate(PROBES):
            ax = axes[0, col]
            table = subset[subset["probe"] == probe].set_index("condition").reindex(order)
            if table[["mean", "sd"]].isna().any().any():
                raise ValueError(f"Missing absolute metric rows for {metric!r}, probe={probe!r}")
            ax.bar(
                x,
                table["mean"].to_numpy(dtype=float),
                yerr=table["sd"].fillna(0.0).to_numpy(dtype=float),
                capsize=3,
            )
            ax.set_xticks(x)
            ax.set_xticklabels([CONDITION_LABELS[c] for c in order], rotation=20, ha="right")
            ax.set_title(probe)
            ax.set_ylim(0.0, 1.02)
            ax.grid(axis="y", alpha=0.25)
            if col == 0:
                ax.set_ylabel(metric)
        fig.suptitle(f"{metric}: absolute condition performance | mean ± SD across training seeds")
        fig.tight_layout()
        path = output_dir / ("absolute_" + metric.lower().replace(" ", "_").replace("/", "_") + ".png")
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[metric] = path
    return saved


ABSOLUTE_PRIMARY_FIGURES = plot_absolute_primary_metrics(PRIMARY_AGG)



## 10. Seed-paired deltas and leave-one-out frequency importance

The `(1,2,4,8)` backbone is the primary reference. For every seed and probe:

- `condition_delta = condition − backbone`;
- for leave-one-out ablations, `importance_loss = backbone − ablated`.

Thus a **positive importance loss** means that removing that backbone frequency reduced the metric.


In [ ]:

if not MASTER_RESULTS.empty:
    backbone_name = str(BACKBONE_CONDITION["condition"])
    key_columns = ["training_seed", "probe"]
    baseline = MASTER_RESULTS[MASTER_RESULTS["condition"] == backbone_name].copy()
    if len(baseline) != len(TRAINING_SEEDS) * len(PROBES):
        raise AssertionError("Backbone baseline is incomplete")

    delta_rows = []
    importance_rows = []
    for metric_name, metric_column in PRIMARY_METRICS.items():
        baseline_metric = baseline[key_columns + [metric_column]].rename(columns={metric_column: "backbone_value"})
        current = MASTER_RESULTS[
            ["condition", "role", "training_seed", "probe", "removed_backbone_frequency_hz", metric_column]
        ].rename(columns={metric_column: "condition_value"})
        merged = current.merge(baseline_metric, on=key_columns, validate="many_to_one")
        merged["delta_vs_backbone"] = merged["condition_value"] - merged["backbone_value"]
        merged["metric"] = metric_name
        merged["metric_column"] = metric_column
        delta_rows.append(merged)

        ablated = merged[merged["role"] == "leave_one_backbone_frequency_out"].copy()
        ablated["importance_loss"] = ablated["backbone_value"] - ablated["condition_value"]
        importance_rows.append(ablated)

    PAIRED_DELTA_BY_SEED = pd.concat(delta_rows, ignore_index=True)
    PAIRED_DELTA_AGG = (
        PAIRED_DELTA_BY_SEED.groupby(["metric", "metric_column", "condition", "role", "probe"], as_index=False)
        .agg(mean_delta=("delta_vs_backbone", "mean"), sd_delta=("delta_vs_backbone", "std"), n=("delta_vs_backbone", "count"))
    )

    ABLATION_IMPORTANCE_BY_SEED = pd.concat(importance_rows, ignore_index=True)
    ABLATION_IMPORTANCE_AGG = (
        ABLATION_IMPORTANCE_BY_SEED.groupby(
            ["metric", "metric_column", "removed_backbone_frequency_hz", "probe"], as_index=False
        )
        .agg(
            mean_importance_loss=("importance_loss", "mean"),
            sd_importance_loss=("importance_loss", "std"),
            n=("importance_loss", "count"),
        )
    )

    PAIRED_DELTA_BY_SEED.to_csv(OUTPUT_ROOT / "paired_delta_vs_backbone_by_seed.csv", index=False)
    PAIRED_DELTA_AGG.to_csv(OUTPUT_ROOT / "paired_delta_vs_backbone_aggregate.csv", index=False)
    ABLATION_IMPORTANCE_BY_SEED.to_csv(OUTPUT_ROOT / "ablation_importance_by_seed.csv", index=False)
    ABLATION_IMPORTANCE_AGG.to_csv(OUTPUT_ROOT / "ablation_importance_aggregate.csv", index=False)

    display(ABLATION_IMPORTANCE_AGG)
else:
    PAIRED_DELTA_BY_SEED = pd.DataFrame()
    PAIRED_DELTA_AGG = pd.DataFrame()
    ABLATION_IMPORTANCE_BY_SEED = pd.DataFrame()
    ABLATION_IMPORTANCE_AGG = pd.DataFrame()


In [ ]:
def plot_ablation_importance(
    aggregate: pd.DataFrame,
    *,
    output_dir: Path = FIGURE_ROOT / "importance_loss",
) -> dict[str, Path]:
    """Backbone minus leave-one-frequency-out performance; positive means the frequency helped."""
    saved: dict[str, Path] = {}
    if aggregate.empty:
        return saved
    output_dir.mkdir(parents=True, exist_ok=True)
    frequencies = list(BACKBONE_FREQUENCIES_HZ)
    x = np.arange(len(frequencies), dtype=float)

    for metric in PRIMARY_METRICS:
        subset = aggregate[aggregate["metric"] == metric]
        fig, axes = plt.subplots(
            1, len(PROBES), figsize=(5.3 * len(PROBES), 4.8), sharey=True, squeeze=False
        )
        for col, probe in enumerate(PROBES):
            ax = axes[0, col]
            table = (
                subset[subset["probe"] == probe]
                .set_index("removed_backbone_frequency_hz")
                .reindex(frequencies)
            )
            if table[["mean_importance_loss", "sd_importance_loss"]].isna().any().any():
                raise ValueError(f"Missing importance rows for {metric!r}, probe={probe!r}")
            ax.bar(
                x,
                table["mean_importance_loss"].to_numpy(dtype=float),
                yerr=table["sd_importance_loss"].fillna(0.0).to_numpy(dtype=float),
                capsize=3,
            )
            ax.axhline(0.0, linewidth=1.0)
            ax.set_xticks(x)
            ax.set_xticklabels([f"{fmt_freq(f)} Hz" for f in frequencies])
            ax.set_title(probe)
            ax.grid(axis="y", alpha=0.25)
            if col == 0:
                ax.set_ylabel(f"Backbone − ablated ({metric})")
        fig.suptitle(
            f"Frequency ablation importance: {metric}\n"
            "Positive values mean masking that backbone frequency reduced performance"
        )
        fig.tight_layout()
        path = output_dir / ("importance_" + metric.lower().replace(" ", "_").replace("/", "_") + ".png")
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[metric] = path
    return saved


IMPORTANCE_FIGURES = plot_ablation_importance(ABLATION_IMPORTANCE_AGG)


## 11. Confusion matrices for every condition

For every mask condition, one figure contains `cnn_s`, `cnn_m`, and `cnn_l` side by side. Each matrix is row-normalized per seed and then averaged across the five training seeds.

These are **experiment-level aggregated confusion matrices** and are intentionally separate from the standard single-run Experiment C confusion plot. Every condition is saved for the archive; compact interpretation should rely first on the importance-loss and per-class recall-loss figures.

In [ ]:

def row_normalized_confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> np.ndarray:
    matrix = np.zeros((n_classes, n_classes), dtype=np.float64)
    for true, pred in zip(y_true.astype(int), y_pred.astype(int), strict=True):
        matrix[true, pred] += 1.0
    row_sum = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, row_sum, out=np.zeros_like(matrix), where=row_sum > 0)


CLASS_LABELS = []
NUM_CLASSES = 0
if FIXED_SPLIT is not None:
    NUM_CLASSES = len(FIXED_SPLIT.class_to_idx)
    CLASS_LABELS = [None] * NUM_CLASSES
    for label, index in FIXED_SPLIT.class_to_idx.items():
        CLASS_LABELS[int(index)] = str(label)


def load_run_confusion(condition_name: str, seed: int, probe: str) -> np.ndarray:
    path = expected_run_dir(condition_name, seed, probe) / "embeddings_test.npz"
    if not path.is_file():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as data:
        y = np.asarray(data["y"])
        pred = np.asarray(data["cnn_pred"])
    return row_normalized_confusion(y, pred, NUM_CLASSES)


CONFUSION_MEAN: dict[tuple[str, str], np.ndarray] = {}
CONFUSION_SD: dict[tuple[str, str], np.ndarray] = {}

if not MASTER_RESULTS.empty:
    for condition in CONDITIONS:
        condition_name = str(condition["condition"])
        for probe in PROBES:
            matrices = np.stack(
                [load_run_confusion(condition_name, seed, probe) for seed in TRAINING_SEEDS],
                axis=0,
            )
            CONFUSION_MEAN[(condition_name, probe)] = matrices.mean(axis=0)
            CONFUSION_SD[(condition_name, probe)] = matrices.std(axis=0, ddof=1)


def plot_condition_confusions() -> dict[str, Path]:
    saved: dict[str, Path] = {}
    if not CONFUSION_MEAN:
        return saved

    for condition in CONDITIONS:
        condition_name = str(condition["condition"])
        fig, axes = plt.subplots(1, len(PROBES), figsize=(5.2 * len(PROBES), 4.8), squeeze=False)
        image = None
        for col, probe in enumerate(PROBES):
            ax = axes[0, col]
            matrix = CONFUSION_MEAN[(condition_name, probe)]
            image = ax.imshow(matrix, vmin=0.0, vmax=1.0, aspect="auto")
            ax.set_title(probe)
            ax.set_xlabel("Predicted class")
            ax.set_ylabel("True class")
            ax.set_xticks(np.arange(NUM_CLASSES))
            ax.set_yticks(np.arange(NUM_CLASSES))
            ax.set_xticklabels(CLASS_LABELS, rotation=90)
            ax.set_yticklabels(CLASS_LABELS)
            if NUM_CLASSES <= 12:
                for i in range(NUM_CLASSES):
                    for j in range(NUM_CLASSES):
                        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=6)

        kept = ", ".join(fmt_freq(v) for v in condition["kept_frequencies_hz"])
        masked = ", ".join(fmt_freq(v) for v in condition["masked_frequencies_hz"]) or "none"
        fig.suptitle(f"{condition['role']} | keep [{kept}] Hz | mask [{masked}] Hz")
        if image is not None:
            fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.78, label="Row-normalized probability")
        path = FIGURE_ROOT / f"confusion_{condition_name}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[condition_name] = path
    return saved


CONFUSION_FIGURES = plot_condition_confusions()



## 12. Per-class recall heatmaps

The diagonal of each row-normalized confusion matrix is per-class recall. These heatmaps show which classes are most affected when a particular backbone frequency is removed.


In [ ]:
def plot_per_class_recall_heatmaps(
    *,
    output_dir: Path = FIGURE_ROOT / "per_class_recall",
) -> dict[str, Path]:
    """Absolute mean recall for every condition."""
    saved: dict[str, Path] = {}
    if not CONFUSION_MEAN:
        return saved
    output_dir.mkdir(parents=True, exist_ok=True)
    order = condition_order()

    for probe in PROBES:
        table = np.stack([np.diag(CONFUSION_MEAN[(condition, probe)]) for condition in order], axis=1)
        fig, ax = plt.subplots(figsize=(max(10, 1.7 * len(order)), max(5, 0.45 * NUM_CLASSES)))
        image = ax.imshow(table, vmin=0.0, vmax=1.0, aspect="auto")
        ax.set_yticks(np.arange(NUM_CLASSES))
        ax.set_yticklabels(CLASS_LABELS)
        ax.set_xticks(np.arange(len(order)))
        ax.set_xticklabels([CONDITION_LABELS[c] for c in order], rotation=20, ha="right")
        ax.set_ylabel("True class")
        ax.set_title(f"{probe}: mean per-class recall across training seeds")
        fig.colorbar(image, ax=ax, label="Recall")
        if NUM_CLASSES <= 12 and len(order) <= 8:
            for i in range(table.shape[0]):
                for j in range(table.shape[1]):
                    ax.text(j, i, f"{table[i, j]:.2f}", ha="center", va="center", fontsize=7)
        fig.tight_layout()
        path = output_dir / f"per_class_recall_absolute_{probe}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[probe] = path
    return saved


def leave_one_out_condition_by_frequency() -> dict[float, str]:
    mapping: dict[float, str] = {}
    for condition in CONDITIONS:
        if str(condition["role"]) != "leave_one_backbone_frequency_out":
            continue
        mapping[float(condition["removed_backbone_frequency_hz"])] = str(condition["condition"])
    return mapping


def plot_per_class_recall_loss_heatmaps(
    *,
    output_dir: Path = FIGURE_ROOT / "per_class_recall",
) -> dict[str, Path]:
    """Backbone recall minus ablated recall for each class and removed frequency."""
    saved: dict[str, Path] = {}
    if not CONFUSION_MEAN:
        return saved
    output_dir.mkdir(parents=True, exist_ok=True)
    backbone_name = str(BACKBONE_CONDITION["condition"])
    ablated_by_frequency = leave_one_out_condition_by_frequency()
    frequencies = [float(v) for v in BACKBONE_FREQUENCIES_HZ]

    missing = [f for f in frequencies if f not in ablated_by_frequency]
    if missing:
        raise ValueError(f"Missing leave-one-out conditions for frequencies: {missing}")

    for probe in PROBES:
        backbone_recall = np.diag(CONFUSION_MEAN[(backbone_name, probe)])
        loss_columns = []
        for frequency in frequencies:
            condition_name = ablated_by_frequency[frequency]
            ablated_recall = np.diag(CONFUSION_MEAN[(condition_name, probe)])
            loss_columns.append(backbone_recall - ablated_recall)
        table = np.stack(loss_columns, axis=1)
        limit = max(float(np.nanmax(np.abs(table))), 1e-12)

        fig, ax = plt.subplots(figsize=(max(7, 1.6 * len(frequencies)), max(5, 0.45 * NUM_CLASSES)))
        image = ax.imshow(table, vmin=-limit, vmax=limit, aspect="auto")
        ax.set_yticks(np.arange(NUM_CLASSES))
        ax.set_yticklabels(CLASS_LABELS)
        ax.set_xticks(np.arange(len(frequencies)))
        ax.set_xticklabels([f"remove {fmt_freq(f)} Hz" for f in frequencies])
        ax.set_ylabel("True class")
        ax.set_title(
            f"{probe}: per-class recall loss from removing each backbone frequency\n"
            "Positive = class recall decreased after masking"
        )
        fig.colorbar(image, ax=ax, label="Backbone recall − ablated recall")
        if NUM_CLASSES <= 12:
            for i in range(table.shape[0]):
                for j in range(table.shape[1]):
                    ax.text(j, i, f"{table[i, j]:+.2f}", ha="center", va="center", fontsize=7)
        fig.tight_layout()
        path = output_dir / f"per_class_recall_loss_{probe}.png"
        fig.savefig(path, dpi=180, bbox_inches="tight")
        plt.show()
        saved[probe] = path
    return saved


PER_CLASS_RECALL_FIGURES = plot_per_class_recall_heatmaps()
PER_CLASS_RECALL_LOSS_FIGURES = plot_per_class_recall_loss_heatmaps()


### Optional: representative standard Experiment C diagnostics

The ablation figures above are the **experiment-level** evidence. This optional cell reuses the repository's standard `visualize_experiment_c()` helper for a small set of representative runs: **full 5-band baseline**, **(1,2,4,8) backbone**, and the **most damaging leave-one-frequency-out condition** according to CNN balanced-accuracy importance for the selected probe.

The cell is disabled by default because the standard Experiment C helper creates many diagnostic figures per run.

In [ ]:
RUN_STANDARD_C_DIAGNOSTICS = False
STANDARD_C_DIAGNOSTIC_PROBE = "cnn_l"
STANDARD_C_DIAGNOSTIC_METRIC_COLUMN = "CNN_test_balanced_accuracy"


def representative_seed_for_mask_condition(condition: str, probe: str) -> int:
    subset = MASTER_RESULTS[
        (MASTER_RESULTS["condition"] == condition) & (MASTER_RESULTS["probe"] == probe)
    ].copy()
    if subset.empty:
        raise ValueError(f"No runs for condition={condition!r}, probe={probe!r}")
    mean_value = float(subset[STANDARD_C_DIAGNOSTIC_METRIC_COLUMN].mean())
    row = subset.loc[(subset[STANDARD_C_DIAGNOSTIC_METRIC_COLUMN] - mean_value).abs().idxmin()]
    return int(row["training_seed"])


def representative_mask_conditions(probe: str) -> list[str]:
    result = [str(FULL_CONDITION["condition"]), str(BACKBONE_CONDITION["condition"])]
    subset = ABLATION_IMPORTANCE_AGG[
        (ABLATION_IMPORTANCE_AGG["metric"] == "CNN balanced accuracy") &
        (ABLATION_IMPORTANCE_AGG["probe"] == probe)
    ].copy()
    if not subset.empty:
        removed = float(subset.loc[subset["mean_importance_loss"].idxmax(), "removed_backbone_frequency_hz"])
        for condition in CONDITIONS:
            if (
                str(condition["role"]) == "leave_one_backbone_frequency_out"
                and np.isclose(float(condition["removed_backbone_frequency_hz"]), removed)
            ):
                result.append(str(condition["condition"]))
                break
    deduped: list[str] = []
    for name in result:
        if name not in deduped:
            deduped.append(name)
    return deduped


STANDARD_C_DIAGNOSTIC_FIGURES: dict[str, dict[str, Path]] = {}
if RUN_STANDARD_C_DIAGNOSTICS and not MASTER_RESULTS.empty:
    from snn.accel_reconstruction_eval.visualization import (
        VisualizationConfig,
        visualize_experiment_c,
    )

    viz_config = VisualizationConfig(dpi=180, show=True, close_after_save=False)
    for condition in representative_mask_conditions(STANDARD_C_DIAGNOSTIC_PROBE):
        seed = representative_seed_for_mask_condition(condition, STANDARD_C_DIAGNOSTIC_PROBE)
        run_dir = expected_run_dir(condition, seed, STANDARD_C_DIAGNOSTIC_PROBE)
        figure_dir = FIGURE_ROOT / "standard_experiment_c" / condition / STANDARD_C_DIAGNOSTIC_PROBE / f"seed_{seed}"
        print(f"Standard Experiment C diagnostics: {condition} | {STANDARD_C_DIAGNOSTIC_PROBE} | seed={seed}")
        STANDARD_C_DIAGNOSTIC_FIGURES[condition] = visualize_experiment_c(
            run_dir,
            figure_dir=figure_dir,
            config=viz_config,
        )
else:
    print("Standard Experiment C representative-run diagnostics are disabled. Set RUN_STANDARD_C_DIAGNOSTICS=True to generate them.")



## 13. Compact report tables


In [ ]:

def format_mean_sd(mean: float, sd: float, digits: int = 3) -> str:
    return f"{mean:.{digits}f} ± {sd:.{digits}f}"


if not PRIMARY_AGG.empty:
    formatted = PRIMARY_AGG.copy()
    formatted["mean ± SD"] = [
        format_mean_sd(float(mean), float(sd))
        for mean, sd in zip(formatted["mean"], formatted["sd"], strict=True)
    ]
    PRIMARY_REPORT = formatted.pivot_table(
        index=["metric", "condition"],
        columns="probe",
        values="mean ± SD",
        aggfunc="first",
    ).reindex(columns=list(PROBES))
    PRIMARY_REPORT.to_csv(OUTPUT_ROOT / "primary_metric_report_table.csv")
    display(PRIMARY_REPORT)

if not ABLATION_IMPORTANCE_AGG.empty:
    formatted = ABLATION_IMPORTANCE_AGG.copy()
    formatted["importance mean ± SD"] = [
        format_mean_sd(float(mean), float(sd))
        for mean, sd in zip(
            formatted["mean_importance_loss"],
            formatted["sd_importance_loss"],
            strict=True,
        )
    ]
    IMPORTANCE_REPORT = formatted.pivot_table(
        index=["metric", "removed_backbone_frequency_hz"],
        columns="probe",
        values="importance mean ± SD",
        aggfunc="first",
    ).reindex(columns=list(PROBES))
    IMPORTANCE_REPORT.to_csv(OUTPUT_ROOT / "ablation_importance_report_table.csv")
    display(IMPORTANCE_REPORT)


## 14. Output checklist and interpretation

The ablation pipeline and Experiment C artifacts are unchanged. This version only expands the visualization layer.

```text
figures/
    absolute_conditions/
        absolute_*.png                    # absolute condition performance; one panel per probe
    importance_loss/
        importance_*.png                  # backbone − ablated; primary ablation figure
    confusion_<condition>.png             # every condition; cnn_s/m/l side by side
    per_class_recall/
        per_class_recall_absolute_*.png
        per_class_recall_loss_*.png       # backbone recall − ablated recall by class/frequency
    standard_experiment_c/                # optional; disabled by default
        <representative condition>/<probe>/seed_<seed>/...
```

### Visualization priority

1. **Frequency importance-loss plots** are the primary ablation figures: positive values mean masking that frequency reduced the metric.
2. Absolute-condition plots show the full 5-band baseline, `(1,2,4,8)` backbone, and all leave-one-out conditions on the same scale.
3. **Per-class recall-loss heatmaps** show which classes depend on 1, 2, 4, or 8 Hz.
4. Every condition retains its seed-aggregated confusion matrix for detailed error inspection.
5. Standard Experiment C visualization is optional and is used only for representative run diagnostics, not for frequency-importance ranking.
